# K-means as a feature generator (model stacking)

_Feature Engineering (Zheng & Casari)_

**Cluster the data, then use cluster membership as new features so a plain linear model can carve nonlinear boundaries.**

You already know k-means as a clustering algorithm: it
       finds $k$ centers and assigns every point to its nearest one. Chapter 7 of Zheng & Casari's
       Feature Engineering for Machine Learning uses k-means for a completely different job &mdash;
       as a feature generator &mdash; the featurization angle on the same algorithm. Instead of
       treating the clusters as the final answer, you treat "which cluster did this point land in" as a
       brand-new feature, then hand that feature to a downstream model.

       This is model stacking: the output of one model (k-means) becomes the input
       features of another (say, logistic regression). The k-means stage does the hard nonlinear work
       of figuring out where on the data manifold each point sits; the linear stage then only has to
       fit a simple weight per region. Together they can separate classes that a plain linear model, fed
       the raw coordinates, could never separate.

---

This notebook is a practice scaffold for the **K-means as a feature generator (model stacking)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — scikit-learn

In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_swiss_roll
from sklearn.preprocessing import OneHotEncoder

# === K-means as a feature generator (Chapter 7) ===
class KMeansFeaturizer(BaseEstimator, TransformerMixin):
    """Fit k-means, then emit ONE-HOT cluster membership as features.
       If a target y is given, append a scaled copy of y so clusters are
       class-purer (semi-supervised). Done on TRAIN ONLY to avoid leakage."""
    def __init__(self, k=100, target_scale=5.0, random_state=None):
        self.k = k
        self.target_scale = target_scale     # 0 -> plain unsupervised k-means
        self.random_state = random_state

    def fit(self, X, y=None):
        if y is None:                        # unsupervised: cluster X as-is
            km = KMeans(n_clusters=self.k, n_init=20,
                        random_state=self.random_state)
            km.fit(X)
        else:                                # semi-supervised: append scaled y
            data_with_target = np.hstack((X, y[:, np.newaxis] * self.target_scale))
            km = KMeans(n_clusters=self.k, n_init=20,
                        random_state=self.random_state)
            km.fit(data_with_target)
            # keep ONLY the centroids' input-space coords for transform time
            km = KMeans(n_clusters=self.k, init=km.cluster_centers_[:, :-1],
                        n_init=1, max_iter=1).fit(X)
        self.km_ = km
        self.ohe_ = OneHotEncoder(sparse_output=False).fit(
            km.predict(X)[:, np.newaxis])
        return self

    def transform(self, X, y=None):
        clusters = self.km_.predict(X)       # nearest centroid -> cluster id
        return self.ohe_.transform(clusters[:, np.newaxis])   # one-hot of cluster

# --- The book's Swiss-roll example ---
X, color = make_swiss_roll(n_samples=1500, noise=0.0, random_state=0)
y = (color > color.mean()).astype(int)       # two classes along the roll

ntrain = 1000
Xtr, ytr, Xte, yte = X[:ntrain], y[:ntrain], X[ntrain:], y[ntrain:]

# 1) Plain logistic regression on the RAW (x, y, z) coordinates -> fails
lr_raw = LogisticRegression().fit(Xtr, ytr)
print('raw-feature accuracy     :', lr_raw.score(Xte, yte))     # ~0.5-0.7

# 2) STACK k-means under logistic regression (model stacking)
kmf = KMeansFeaturizer(k=100, target_scale=5.0,
                       random_state=0).fit(Xtr, ytr)             # fit on TRAIN
Ztr = kmf.transform(Xtr)                                         # cluster features
Zte = kmf.transform(Xte)
lr_clu = LogisticRegression().fit(Ztr, ytr)
print('cluster-feature accuracy :', lr_clu.score(Zte, yte))     # ~0.95+

# Distance-feature variant: use km.transform(X) (distances to all k centroids)
# instead of one-hot of km.predict(X) for a soft, radial-basis-like encoding.

## Visualize the data & results

_On real data with nonlinear structure (load_digits projected to 2-D, where a linear model struggles), does feeding k-means CLUSTER features to logistic regression beat feeding the RAW features?_

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Real data: the hard 4-vs-9 digit pair (curved, not linearly separable in 2-D)
d = load_digits()
mask = np.isin(d.target, [4, 9])
X, y = d.data[mask], (d.target[mask] == 9).astype(int)

# Project to 2-D so a linear model genuinely struggles (nonlinear structure)
X2 = PCA(n_components=2, random_state=0).fit_transform(StandardScaler().fit_transform(X))
Xtr, Xte, ytr, yte = train_test_split(X2, y, test_size=0.4, random_state=0)

def lr_acc(Ztr, Zte):
    return LogisticRegression(max_iter=2000).fit(Ztr, ytr).score(Zte, yte)

# RAW 2-D coordinates -> linear model struggles
print('raw      :', round(lr_acc(Xtr, Xte), 2))                 # -> 0.46

# k-means CLUSTER features (one-hot of nearest centroid), fit on TRAIN only
for k in (8, 40, 100):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Xtr)
    ohe = OneHotEncoder(sparse_output=False).fit(km.predict(Xtr)[:, None])
    Ztr = ohe.transform(km.predict(Xtr)[:, None])
    Zte = ohe.transform(km.predict(Xte)[:, None])
    print('k=%-3d    :' % k, round(lr_acc(Ztr, Zte), 2))        # 0.61 / 0.83 / 0.91

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You wrap logistic regression around k-means cluster features on a curved (two-moons-like) dataset, but accuracy barely improves over raw features. You used $k=3$ clusters. What is the most likely fix, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall that the linear model gets one weight per cluster, so it can only output one constant per cell. — _With just 3 cells over interleaving arcs, each cell still contains a mix of both classes._
- Increase $k$ substantially &mdash; tens of clusters &mdash; so each cell sits on essentially one class. — _Chapter 7's key rule: you need ENOUGH clusters to tile the manifold densely, otherwise the per-cell constant can't separate the classes._
- Re-fit k-means on the training set only with the larger $k$, then re-encode and re-train the downstream model. — _More, smaller cells give the linear model the resolution to bend its boundary along the manifold._

**Answer:** The fix is to use many more clusters. With $k=3$, each Voronoi cell straddles both arcs, so the one prediction per cell can't separate them &mdash; the model underfits. Raising $k$ into the tens (or more) tiles the manifold densely enough that each cell is essentially one class, and the linear model recovers a piecewise approximation of the curved boundary.

</details>

**Problem 2.** To boost the cluster features you append the class label $y$ (scaled) as an extra coordinate before running k-means. On the test set, accuracy looks suspiciously perfect. What went wrong, and what is the correct procedure?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Notice that the clustering used $y$, and the SAME k-means was applied to the test points. — _If the test labels helped place the centroids (or were used to assign test points), the features encode the answer directly._
- Identify this as data leakage from the target into the features. — _Label information has bled into the feature definition, inflating measured performance._
- Cluster OUT-OF-FOLD: fit the label-augmented k-means on the TRAINING fold only, freeze the centroids, then assign validation/test points using their input features alone (no $y$). — _This keeps the semi-supervised benefit on train while preventing test labels from touching the features._

**Answer:** It is target leakage: clustering with $y$ and then applying it to test points lets the test labels define the features. The book's semi-supervised trick is fine, but it must be done out-of-fold &mdash; fit the label-augmented k-means on training data only, freeze the centroids, and assign held-out points by their inputs alone. The label is never available at assignment time for the data you score.

</details>

**Problem 3.** Compare k-means distance features ($\phi_{\text{dist}}$, scikit-learn's KMeans.transform) with one-hot cluster membership ($\phi_{\text{onehot}}$) as inputs to a linear model. When might the distance encoding help, and what is its cost?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note that one-hot is a hard 0/1: a point belongs entirely to one cluster, so the boundary between cells is a hard jump. — _The downstream model gets a piecewise-CONSTANT function — fine far from boundaries, crude near them._
- Note that distances are soft: a point near two centroids has small distances to both, so the model sees a smooth blend. — _This gives a piecewise-LINEAR (radial-basis-like) surface that interpolates between cells, often smoother near boundaries._
- Weigh the cost: distances are dense (all $k$ values nonzero) versus the sparse single 1 of one-hot. — _Dense features are larger and can need scaling, whereas one-hot stays sparse and very cheap._

**Answer:** One-hot gives a piecewise-constant model: cheap and sparse, but it snaps hard at cell boundaries. Distance features (KMeans.transform) give a soft, radial-basis-like encoding so the linear model can interpolate between clusters &mdash; helpful when the true boundary cuts across cells. The cost is that distances are dense (all $k$ coordinates nonzero) and benefit from scaling, versus one-hot's single 1.

</details>